# Parquet Pivot Analysis

This notebook is the central orchestration layer for tariff parquet analysis.
Helper modules in the same folder handle path resolution, parquet loading, pivots,
and annual contract cost calculations for the new storage schema.


In [1]:
from pathlib import Path
import importlib
import sys
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'storage_paths.py').exists():
    candidate = Path('Attempt3/Tariff_preprocesssing/new_organization_tariff_data/contract cost calculations').resolve()
    if (candidate / 'storage_paths.py').exists():
        NOTEBOOK_DIR = candidate

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

importlib.invalidate_caches()
import storage_paths
importlib.reload(storage_paths)
resolve_storage_dir = getattr(storage_paths, 'resolve_storage_dir', None)
if resolve_storage_dir is None:
    resolve_storage_dir = storage_paths.resolve_storage_output_dir
resolve_profile_path = storage_paths.resolve_profile_path
from parquet_loader import load_storage_frames, build_contract_catalog
from tariff_pivots import build_usage_analysis_table, build_fee_analysis_table, provider_year_overview
from annual_contract_costs import calculate_annual_contract_costs

storage_dir = resolve_storage_dir()
profile_path = resolve_profile_path()
print(f'Storage dir: {storage_dir}')
print(f'Profile path: {profile_path}')


Storage dir: C:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\Tariff_preprocesssing\new_organization_tariff_data\storage files output
Profile path: C:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\Tariff_preprocesssing\new_organization_tariff_data\contract cost calculations\4c6a58e2-f666-4d3d-b8c0-55d32a781314.json


In [2]:
frames = load_storage_frames(storage_dir)
frame_summary = pd.DataFrame([
    {'name': 'contracts_fixed', 'rows': len(frames.contracts_fixed), 'columns': len(frames.contracts_fixed.columns)},
    {'name': 'contracts_variable', 'rows': len(frames.contracts_variable), 'columns': len(frames.contracts_variable.columns)},
    {'name': 'fixed_usage', 'rows': len(frames.fixed_usage), 'columns': len(frames.fixed_usage.columns)},
    {'name': 'variable_usage', 'rows': len(frames.variable_usage), 'columns': len(frames.variable_usage.columns)},
    {'name': 'fixed_fees', 'rows': len(frames.fixed_fees), 'columns': len(frames.fixed_fees.columns)},
    {'name': 'variable_fees', 'rows': len(frames.variable_fees), 'columns': len(frames.variable_fees.columns)},
])
print(frame_summary.to_string(index=False))


              name  rows  columns
   contracts_fixed  3649       24
contracts_variable  2520       24
       fixed_usage  8978       11
    variable_usage  6228       11
        fixed_fees  7298       10
     variable_fees  5040       10


In [3]:
contract_catalog = build_contract_catalog(frames)
print(f'Contract catalog rows: {len(contract_catalog):,}')
print(f'Providers: {contract_catalog["provider_name"].nunique():,}')
print(f'Offer keys: {contract_catalog["offer_key"].nunique():,}')
print()
print(contract_catalog[[
    'provider_name',
    'contract_base_name',
    'contract_duration_label',
    'contract_type',
    'meter_type',
    'snapshot_month',
    'contract_snapshot_key',
    'contract_base_key',
    'offer_key',
]].head(12).to_string(index=False))


Contract catalog rows: 6,169
Providers: 47
Offer keys: 440

provider_name contract_base_name contract_duration_label contract_type meter_type snapshot_month                                                                  contract_snapshot_key     contract_base_key                                                offer_key
    AllureNRG        Vaste Prijs           Vast (3 jaar)         fixed     double        2025-04 allurenrg|fixed|double|vaste_prijs_3_jaar_zonder_zonnepanelen|vast_3_jaar|none|2025-04 allurenrg|vaste_prijs allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|double
    AllureNRG        Vaste Prijs           Vast (3 jaar)         fixed     single        2025-04    allurenrg|fixed|single|vaste_prijs_3_jaar_met_zonnepanelen|vast_3_jaar|none|2025-04 allurenrg|vaste_prijs allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|single
    AllureNRG        Vaste Prijs           Vast (2 jaar)         fixed     double        2025-04    allurenrg|fixed|double|vaste_prijs_2_jaar_met_zonne

In [4]:
usage_table = build_usage_analysis_table(frames)
fee_table = build_fee_analysis_table(frames)
overview = provider_year_overview(frames, provider_names=['Greenchoice', 'Essent', 'Vattenfall'], year=2025)

print('Usage table sample:')
print(usage_table[[
    'provider_name', 'contract_type', 'commodity', 'tariff_band', 'snapshot_month', 'rate', 'offer_key'
]].head(10).round(6).to_string(index=False))
print()
print('Fee table sample:')
print(fee_table[[
    'provider_name', 'contract_type', 'fee_component', 'snapshot_month', 'annual_amount', 'offer_key'
]].head(10).round(2).to_string(index=False))
print()
print('Provider-year contract overview:')
print(overview['contracts'][[
    'provider_name', 'contract_type', 'contract_base_name', 'contract_duration_label', 'meter_type', 'snapshot_month'
]].head(15).to_string(index=False))
print()
print('Usage pivot sample:')
print(overview['usage_pivot'].head(10).round(6).to_string())
print()
print('Fee pivot sample:')
print(overview['fee_pivot'].head(10).round(2).to_string())


Usage table sample:
provider_name contract_type   commodity tariff_band snapshot_month   rate                                                offer_key
    AllureNRG         fixed electricity        peak        2025-04 0.2844 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|double
    AllureNRG         fixed electricity     offpeak        2025-04 0.2844 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|double
    AllureNRG         fixed         gas      single        2025-04 1.2985 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|double
    AllureNRG         fixed electricity      single        2025-04 0.2844 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|single
    AllureNRG         fixed         gas      single        2025-04 1.2985 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|single
    AllureNRG         fixed electricity        peak        2025-04 0.2892 allurenrg|fixed|allurenrg|vaste_prijs|vast_2_jaar|double
    AllureNRG         fixed electricity     offpeak        2025

In [5]:
annual_results = calculate_annual_contract_costs(
    storage_dir=storage_dir,
    profile_path=profile_path,
    annual_gas_m3=800.0,
    year=2025,
)

print('Consumption summary used for annual costs:')
print(annual_results['consumption_summary'].round(3).to_string(index=False))
print()
print('Annual 2025 cost summary by contract type:')
print(annual_results['summary_by_contract_type'].round(2).to_string(index=False))
print()
print('Feed-in cost coverage:')
print(f"  Fixed offers with feed-in cost: {(annual_results['fixed_annual_offer_costs']['feed_in_cost_eur'] > 0).sum()}")
print(f"  Variable offers with feed-in cost: {(annual_results['variable_annual_offer_costs']['feed_in_cost_eur'] > 0).sum()}")
print()
print('Fixed tariff matrix sample:')
print(annual_results['fixed_tariff_matrix'][[
    'provider_name', 'contract_base_name', 'meter_type', 'electricity_peak', 'electricity_offpeak', 'gas'
]].head(10).round(6).to_string(index=False))
print()
print('Fixed usage matrix sample:')
print(annual_results['fixed_usage_matrix'].head(5).round(3).to_string(index=False))
print()
print('Variable monthly tariff matrix sample:')
print(annual_results['variable_monthly_tariff_matrix'].head(12).round(6).to_string(index=False))
print()
print('Variable monthly usage matrix sample:')
print(annual_results['variable_monthly_usage_matrix'].head(12).round(3).to_string(index=False))
print()
print('Variable weighted annual tariff matrix sample:')
print(annual_results['variable_weighted_tariff_matrix'].head(12).round(6).to_string(index=False))


Consumption summary used for annual costs:
 annual_import_kwh  annual_feedin_kwh  annual_net_load_kwh  annual_gas_m3  annual_peak_import_kwh  annual_offpeak_import_kwh
          6500.003           4000.004               2500.0          800.0                3553.622                   2946.381

Annual 2025 cost summary by contract type:
contract_type  offers  providers  avg_total_annual_cost_eur  min_total_annual_cost_eur  max_total_annual_cost_eur
        fixed     227         26                    3785.33                    2684.79                    5339.89
     variable     213         47                    4489.07                    2747.96                   14427.90

Feed-in cost coverage:
  Fixed offers with feed-in cost: 213
  Variable offers with feed-in cost: 200

Fixed tariff matrix sample:
 provider_name                               contract_base_name meter_type  electricity_peak  electricity_offpeak    gas
     AllureNRG                                      Vaste Prijs     

In [ ]:
fixed_annual = annual_results['fixed_annual_offer_costs'].copy()
variable_annual = annual_results['variable_annual_offer_costs'].copy()
fixed_display_cols = [
    'provider_name',
    'contract_base_name',
    'contract_duration_label',
    'meter_type',
    'electricity_cost_eur',
    'gas_cost_eur',
    'fixed_fee_cost_eur',
    'feed_in_cost_eur',
    'total_annual_cost_eur',
]
variable_display_cols = [
    'provider_name',
    'contract_base_name',
    'contract_duration_label',
    'meter_type',
    'months_available',
    'electricity_cost_eur',
    'gas_cost_eur',
    'fixed_fee_cost_eur',
    'feed_in_cost_eur',
    'total_annual_cost_eur',
]

print('Fixed annual offer costs for 2025:')
print(fixed_annual[fixed_display_cols].head(20).round(2).to_string(index=False))
print()
print('Fixed offers with non-zero feed-in costs:')
print(fixed_annual[fixed_annual['feed_in_cost_eur'] > 0].sort_values('feed_in_cost_eur', ascending=False)[fixed_display_cols].head(20).round(2).to_string(index=False))
print()
print('Variable annual offer costs for 2025:')
print(variable_annual[variable_display_cols].head(20).round(2).to_string(index=False))
print()
print('Variable offers with non-zero feed-in costs:')
print(variable_annual[variable_annual['feed_in_cost_eur'] > 0].sort_values('feed_in_cost_eur', ascending=False)[variable_display_cols].head(20).round(2).to_string(index=False))


Fixed annual offer costs for 2025:
   provider_name             contract_base_name contract_duration_label meter_type  electricity_cost_eur  gas_cost_eur  fixed_fee_cost_eur  feed_in_cost_eur  total_annual_cost_eur
Coolblue Energie                         Direct          Vast (<1 jaar)     single               1481.35        987.68              215.76              0.00                2684.79
Coolblue Energie                         Direct          Vast (<1 jaar)     double               1481.72        987.68              215.76              0.00                2685.15
Coolblue Energie Coolblue Energie Vast (3 jaar)           Vast (3 jaar)     single               1589.25       1079.20              215.98              0.00                2884.44
Coolblue Energie Coolblue Energie Vast (3 jaar)           Vast (3 jaar)     double               1591.97       1079.20              215.98              0.00                2887.15
Coolblue Energie Coolblue Energie Vast (1 jaar)           Vast (1

: 